# 01 - Data Cleaning
Load the six TMDB CSVs, inspect them, clean them, and load them into a SQLite database.

In [45]:
import pandas as pd
import numpy as np 

# Load all 6 CSVs
movies = pd.read_csv("data/movies.csv")
genres = pd.read_csv("data/genres.csv")
movie_genres = pd.read_csv("data/movie_genres.csv")
cast = pd.read_csv("data/cast.csv")
crew = pd.read_csv("data/crew.csv")
movie_keywords = pd.read_csv("data/movie_keywords.csv")

In [46]:
tables = {
    "movies": movies, "genres": genres, "movie_genres": movie_genres,
    "cast": cast, "crew": crew, "movie_keywords": movie_keywords
}

for name, df in tables.items():
    print("=" * 50)
    print(name, "-> shape:", df.shape)
    print(df.dtypes)
    print("Null counts:")
    print(df.isnull().sum())
    print("Duplicate rows:", df.duplicated().sum())



movies -> shape: (2509, 12)
movie_id               int64
title                    str
original_language        str
release_date             str
runtime                int64
budget               float64
revenue              float64
popularity           float64
vote_average         float64
vote_count             int64
status                   str
overview                 str
dtype: object
Null counts:
movie_id             0
title                0
original_language    0
release_date         0
runtime              0
budget               0
revenue              0
popularity           0
vote_average         0
vote_count           0
status               0
overview             0
dtype: int64
Duplicate rows: 6
genres -> shape: (19, 2)
genre_id      int64
genre_name      str
dtype: object
Null counts:
genre_id      0
genre_name    0
dtype: int64
Duplicate rows: 0
movie_genres -> shape: (6888, 2)
movie_id    int64
genre_id    int64
dtype: object
Null counts:
movie_id    0
genre_id    0
dtype: int6

In [47]:
movies.head()

,movie_id,title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,status,overview
0,157336,Interstellar,en,2014-11-05,169,165000000.0,7.466067e+08,73.0452,8.484,40566,Released,The adventures of a group of explorers who mak...
1,27205,Inception,en,2010-07-15,148,160000000.0,8.390306e+08,55.8497,8.400,39727,Released,"Cobb, a skilled thief who commits corporate es..."
2,24428,The Avengers,en,2012-04-25,143,220000000.0,1.518816e+09,72.2515,8.062,38993,Released,When an unexpected enemy emerges and threatens...
3,155,The Dark Knight,en,2008-07-16,152,185000000.0,1.004558e+09,60.5510,8.534,36255,Released,Batman raises the stakes in his war on crime. ...
4,19995,Avatar,en,2009-12-16,162,237000000.0,2.923706e+09,48.1996,7.610,34348,Released,"In the 22nd century, a paraplegic Marine is di..."


In [48]:
genres.head()

,genre_id,genre_name
0,28,Action
1,12,Adventure
2,16,Animation
3,35,Comedy
4,80,Crime


In [49]:
movie_genres.head()

,movie_id,genre_id
0,157336,12
1,157336,18
2,157336,878
3,27205,28
4,27205,878


In [50]:
cast.head()

,cast_row_id,movie_id,person_id,actor_name,character_name,cast_order
0,1,157336,10297,Matthew McConaughey,Cooper,0
1,2,157336,1813,Anne Hathaway,Brand,1
2,3,157336,3895,Michael Caine,Professor Brand,2
3,4,157336,83002,Jessica Chastain,Murph,3
4,5,157336,1893,Casey Affleck,Tom,4


In [51]:
crew.head()

,crew_row_id,movie_id,person_id,person_name,job,department
0,1,157336,527,Jonathan Nolan,Writer,Writing
1,2,157336,525,Christopher Nolan,Writer,Writing
2,3,157336,525,Christopher Nolan,Director,Directing
3,4,27205,525,Christopher Nolan,Writer,Writing
4,5,27205,525,Christopher Nolan,Director,Directing


In [52]:
movie_keywords.head()

,movie_id,keyword_id,keyword_name
0,157336,1612,spacecraft
1,157336,4776,race against time
2,157336,310,artificial intelligence (a.i.)
3,157336,1432,nasa
4,157336,1521,time warp


## Check hidden placeholders in movies 

In [53]:

print("Zero budget:", (movies["budget"] == 0).sum())
print("Zero revenue:", (movies["revenue"] == 0).sum())
print("Zero runtime:", (movies["runtime"] == 0).sum())
print("Empty overview:", (movies["overview"].str.strip() == "").sum())
print(movies["status"].value_counts())
print(movies.describe())

Zero budget: 90
Zero revenue: 102
Zero runtime: 0
Empty overview: 0
status
Released    2509
Name: count, dtype: int64
           movie_id      runtime        budget       revenue   popularity  \
count  2.509000e+03  2509.000000  2.509000e+03  2.509000e+03  2509.000000   
mean   1.848254e+05   114.420885  5.863972e+07  2.141241e+08    13.691851   
std    2.571634e+05    21.412944  6.232276e+07  2.716873e+08    18.096247   
min    5.000000e+00    45.000000  0.000000e+00  0.000000e+00     0.017000   
25%    4.935000e+03    99.000000  1.450000e+07  4.738629e+07     7.449900   
50%    3.953800e+04   111.000000  3.800000e+07  1.279000e+08    10.393400   
75%    3.304590e+05   126.000000  8.250000e+07  2.713333e+08    14.863900   
max    1.368170e+06   242.000000  6.588000e+08  2.923706e+09   438.475700   

       vote_average    vote_count  
count   2509.000000   2509.000000  
mean       6.941640   6194.598246  
std        0.711199   4963.789824  
min        2.945000    475.000000  
25%     

vote_count has a minimum of 475, so every movie has plenty of votes. That means ratings in this dataset are fairly reliable. 

## Remove exact duplicates 

In [54]:
movies = movies.drop_duplicates(keep="first")
movie_genres = movie_genres.drop_duplicates(keep="first")
cast = cast.drop_duplicates(keep="first")
crew = crew.drop_duplicates(keep="first")
movie_keywords = movie_keywords.drop_duplicates(keep="first")


print("movie_id unique:", movies["movie_id"].is_unique)
print("cast_row_id unique:", cast["cast_row_id"].is_unique)
print("person_id unique:", cast["person_id"].is_unique)
print("crew_row_id unique:", crew["crew_row_id"].is_unique)
print("keyword_id unique:", movie_keywords["keyword_id"].is_unique)

movie_id unique: True
cast_row_id unique: True
person_id unique: False
crew_row_id unique: True
keyword_id unique: False


## Fix data types and nulls

In [55]:
movies["release_date"] = pd.to_datetime(movies["release_date"], format="%Y-%m-%d", errors="coerce")


In [56]:
print("Unparseable dates:", movies["release_date"].isnull().sum())

Unparseable dates: 0


In [57]:
print(cast["character_name"].isnull().sum())

11


In [58]:
cast["character_name"] = cast["character_name"].fillna("Unknown")

In [59]:
print(cast["character_name"].isnull().sum())

0


## Remove orphaned rows

In [60]:
for name, df in {"movie_genres": movie_genres, "cast": cast,
                 "crew": crew, "movie_keywords": movie_keywords}.items():
    orphans = (~df["movie_id"].isin(movies["movie_id"])).sum()
    print(name, "orphan rows:", orphans)


movie_genres orphan rows: 0
cast orphan rows: 0
crew orphan rows: 0
movie_keywords orphan rows: 0


In [61]:
movie_genres = movie_genres[movie_genres["movie_id"].isin(movies["movie_id"])]
movie_genres = movie_genres[movie_genres["genre_id"].isin(genres["genre_id"])]
cast = cast[cast["movie_id"].isin(movies["movie_id"])]
crew = crew[crew["movie_id"].isin(movies["movie_id"])]
movie_keywords = movie_keywords[movie_keywords["movie_id"].isin(movies["movie_id"])]


 ## Final check 

In [62]:
for name, df in {"movies": movies, "genres": genres, "movie_genres": movie_genres,
                 "cast": cast, "crew": crew, "movie_keywords": movie_keywords}.items():
    print(name, df.shape, "| nulls:", df.isnull().sum().sum(), "| dups:", df.duplicated().sum())


movies (2503, 12) | nulls: 0 | dups: 0
genres (19, 2) | nulls: 0 | dups: 0
movie_genres (6878, 2) | nulls: 0 | dups: 0
cast (24902, 6) | nulls: 0 | dups: 0
crew (7289, 6) | nulls: 0 | dups: 0
movie_keywords (41681, 3) | nulls: 0 | dups: 0


## Create the database and load the tables 

In [63]:


from sqlalchemy import create_engine

engine = create_engine("sqlite:///data/movies.db")

movies.to_sql("movies", engine, if_exists="replace", index=False)
genres.to_sql("genres", engine, if_exists="replace", index=False)
movie_genres.to_sql("movie_genres", engine, if_exists="replace", index=False)
cast.to_sql("cast", engine, if_exists="replace", index=False)
crew.to_sql("crew", engine, if_exists="replace", index=False)
movie_keywords.to_sql("movie_keywords", engine, if_exists="replace", index=False)

41681

## test query

In [64]:
test_query = """
SELECT m.title, g.genre_name
FROM movies m
JOIN movie_genres mg ON m.movie_id = mg.movie_id
JOIN genres g ON mg.genre_id = g.genre_id
LIMIT 5
"""
print(pd.read_sql(test_query, engine))

          title       genre_name
0  Interstellar        Adventure
1  Interstellar            Drama
2  Interstellar  Science Fiction
3     Inception        Adventure
4     Inception           Action


## row counts in the database 

In [65]:
for t in ["movies", "genres", "movie_genres", "cast", "crew", "movie_keywords"]:
    n = pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", engine)["n"][0]
    print(t, n)

movies 2503
genres 19
movie_genres 6878
cast 24902
crew 7289
movie_keywords 41681


## Insights

1. Duplicates
The genres table had no duplicates. Exact duplicate rows were found and removed in the other five tables: movies (6), movie_genres (10), cast (15), crew (8) and movie_keywords (20).

2. Orphaned rows
None were found. Every movie_id in movie_genres, cast, crew and movie_keywords exists in movies, and every genre_id in movie_genres exists in genres.

3. Missing values
Only cast.character_name had nulls (11), and these were filled with "Unknown". All other tables had no nulls.

4. Data types
release_date was converted from text to datetime. All ID columns were already integers.

5. Zero budget and revenue
90 movies have a budget of 0 and 102 have a revenue of 0. In TMDB, 0 means the value is unknown, not that the movie was free. These values were left unchanged for now and will be excluded from financial analysis (profit, ROI) in the Pandas section.

6. Database load
The cleaned tables (2,503 movies) were loaded into the SQLite database data/movies.db.

Recommendation
Always filter out zero budget and revenue values before calculating profit or ROI, otherwise the results will be misleading.

